In [ ]:
import pandas as pd
import csv
import graphviz
import math
import numpy as np
import random
from keras.models import Sequential
from keras.layers import Dense, Embedding
from keras.layers import LSTM
from matplotlib import pyplot as plt
from keras.preprocessing.text import Tokenizer
from sklearn.model_selection import KFold
from keras.layers import Dropout
from keras.preprocessing.sequence import pad_sequences
from keras.utils.vis_utils import plot_model
import pydotplus as pydot

In [ ]:
import csv
from google.colab import files
uploaded = files.upload()

In [ ]:
Original = pd.read_csv("Helpdesk_Log_Original.csv")
Original.shape

In [ ]:
Variant = pd.read_csv("Variant_1.csv")
Variant.shape

In [ ]:
# Data Prep (original Log)

_ncols = ('X', 'Y')
_activeCase = "NULL"
maindfObj = pd.DataFrame([], columns=_ncols)

_tempxy = []

def create_input_output(xy, case_id):
    global maindfObj
    # Define Empty List
    values = []
    xList = []
    #print(xy)
    values.append(("NULL", xy[0]))
    i = 0
    while i < len(xy):
        try:
            xList = xy[0: i+1]
            xList.insert(0, "NULL")
            values.append((xList, xy[i + 1]))
        except:
            xList = xy[0: i+1]
            xList.insert(0, "NULL")
            values.append((xList, "END"))
        i = i + 1

    #print("values: ",values)

    subdfObj = pd.DataFrame(values, columns=_ncols)
    maindfObj = maindfObj.append(subdfObj)


for index, row in Original.iterrows():
      if 'case' in row and (row['case'] == _activeCase or _activeCase == "NULL"):
          concatenatedString = row['event']
          _tempxy.append(concatenatedString)
          _activeCase = row['case']

      else:
        create_input_output(_tempxy, _activeCase)
        _activeCase = row['case']
        _tempxy.clear()
        concatenatedString = row['event']
        _tempxy.append(concatenatedString)

Prep_data = maindfObj

tokenizer = Tokenizer(filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n')
tokenizer.fit_on_texts(Prep_data['X'])
X = tokenizer.texts_to_sequences(Prep_data['X'])
word_index = tokenizer.word_index
print(word_index)#
print('Found %s unique tokens.' % len(word_index))

from keras.preprocessing.sequence import pad_sequences

#MAX_SEQUENCE_LENGTH = 6
X = pad_sequences(X)
print(X.shape)

Y = pd.get_dummies(Prep_data['Y'])
print(Y.shape)


In [ ]:
# Model Training


MAX_NB_WORDS =20
EMBEDDING_DIM =32

model = Sequential()
model.add(Embedding(MAX_NB_WORDS, EMBEDDING_DIM, input_length=X.shape[1]))
model.add(LSTM(100, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(Y.shape[1], activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()
plot_model(model, to_file='model.png', show_shapes=True)
print('Training...')

history = model.fit(X, Y,  epochs=50, batch_size=15, verbose=2)

In [ ]:
Original_pred = model.predict(X)
print(Original_pred[0].size)
colName = [];
for i in Y:
    colName.append(i)
dfObj = pd.DataFrame(list(np.round(Original_pred*100, decimals=0)), columns = colName)
#dfObj = dfObj.drop_duplicates()
#print(colName)

In [ ]:
Seq_Series=Prep_data.X.apply(pd.Series)

dfObj.reset_index(drop=True, inplace=True)
Seq_Series.reset_index(drop=True, inplace=True)
df_new = pd.concat([Seq_Series, dfObj], axis=1)

df_new.to_csv(path_or_buf= "Originalpred.csv", index=True)
#from google.colab import files
#files.download('./Originalpred.csv')

In [ ]:
# Generate complete graph for original log
#

In [ ]:
# Data Prep (Variant log)

_ncols = ('X1', 'Y1')
_activeCase = "NULL"
maindfObj = pd.DataFrame([], columns=_ncols)

_tempxy = []

def create_input_output(xy, case_id):
    global maindfObj
    # Define Empty List
    values = []
    xList = [];
    #print(xy)
    values.append(("NULL", xy[0]))
    i = 0
    while i < len(xy):
        try:
            xList = xy[0: i+1]
            xList.insert(0, "NULL")
            values.append((xList, xy[i + 1]))
        except:
            xList = xy[0: i+1]
            xList.insert(0, "NULL")
            values.append((xList, "END"))
        i = i + 1

    #print("values: ",values)

    subdfObj = pd.DataFrame(values, columns=_ncols)
    maindfObj = maindfObj.append(subdfObj)


for index, row in Variant.iterrows():
      if 'case' in row and (row['case'] == _activeCase or _activeCase == "NULL"):
          concatenatedString = row['event']
          _tempxy.append(concatenatedString)
          _activeCase = row['case']

      else:
        create_input_output(_tempxy, _activeCase)
        _activeCase = row['case']
        _tempxy.clear()
        concatenatedString = row['event']
        _tempxy.append(concatenatedString)

Prep_data = maindfObj

tokenizer = Tokenizer(filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n')
tokenizer.fit_on_texts(Prep_data['X1'])
X1 = tokenizer.texts_to_sequences(Prep_data['X1'])
word_index = tokenizer.word_index
print(word_index)#
print('Found %s unique tokens.' % len(word_index))

from keras.preprocessing.sequence import pad_sequences

#MAX_SEQUENCE_LENGTH = 15
X1 = pad_sequences(X1)
print(X1.shape)

Y1 = pd.get_dummies(Prep_data['Y1'])
print(Y1.shape)


In [ ]:
Variant_pred = model.predict(X1)
print(Variant_pred[0].size)
colName = [];
for i in Y1:
    colName.append(i)
dfObj = pd.DataFrame(list(np.round(Variant_pred*100, decimals=0)), columns = colName)
#dfObj = dfObj.drop_duplicates()
#print(colName)

Seq_Series=Prep_data.X1.apply(pd.Series)

dfObj.reset_index(drop=True, inplace=True)
Seq_Series.reset_index(drop=True, inplace=True)
df_new = pd.concat([Seq_Series, dfObj], axis=1)

df_new.to_csv(path_or_buf= "Variant3_pred.csv", index=True)
#from google.colab import files
#files.download('./Variant3_pred.csv')

In [ ]:
File1 = pd.read_csv("/content/Originalpred.csv")

headers = list(File1.columns.values)
File1 = File1.drop([headers[0]], axis=1)

In [ ]:
File2 = pd.read_csv("/content/Variant3_pred.csv")

headers = list(File2.columns.values)
File2 = File2.drop([headers[0]], axis=1)

print(File1.shape, File2.shape)

In [ ]:
class helper:
  def __init__(self):
    self.i= 0
  def grabEventsFromHeader(self, header):
    evs = []
    c = 0
    for ev in header:
      if c > 0:
        try:
          num = int(ev)
        except:
          evs.append(ev)
      c = c + 1
    return evs # return all events
  def rowIsFirst(self, row, activities, headers):
    foundValues = [];
    for i in range(0,len(headers)):
      val = headers[i]
      if val not in activities:
        if not row[val]:
          return True
        else:
          if len(foundValues) > 0:
            return False;
          else:
            foundValues.append(row[val])
    return False;

  def divideMatrix(self, matrix): # use the columns to divide the matrix
    headers = list(matrix.columns.values)
    leftHeaders = []
    rightHeaders = []
    for i in range(0,len(headers)):
      ev = headers[i]
      try:
          num = int(ev)
          leftHeaders.append(ev)
      except:
          rightHeaders.append(ev)
    leftData = matrix[leftHeaders]
    rightData = matrix[rightHeaders]
    return [leftData, rightData]

In [ ]:
class resultGraphing:
  def __init__(self):
    self.i= 0
  def decomposeResult(self, results):
    helper_ = helper()
    headers = list(results.columns.values)
    events = helper_.grabEventsFromHeader(headers)
    matrices = []
    matricesLeft = []
    lastIndex = -1
    totalBegin = 0
    ind = 0
    newMatrix = helper_.divideMatrix(results)
    for index, row in results.iterrows():
      ind = index
      if helper_.rowIsFirst(row, events, headers):
        totalBegin = totalBegin + 1
        if lastIndex > -1:
          if lastIndex == 0:
            sequenceList = newMatrix[1].iloc[lastIndex:index, :]
            sequenceListLeft = newMatrix[0].iloc[lastIndex:index, :]
            matrices.append(sequenceList)
            matricesLeft.append(sequenceListLeft)
          else:
            sequenceList = newMatrix[1].iloc[lastIndex+1:index, :]
            sequenceListLeft = newMatrix[0].iloc[lastIndex+1:index, :]
            matrices.append(sequenceList)
            matricesLeft.append(sequenceListLeft)
        if index < 1:
          lastIndex = index
        else:
          lastIndex = index - 1
    sequenceList = newMatrix[1].iloc[lastIndex+1:ind+1, :]
    sequenceListLeft = newMatrix[0].iloc[lastIndex+1:ind+1, :]
    matrices.append(sequenceList)
    matricesLeft.append(sequenceListLeft)
    return matrices, matricesLeft

  def getEventSequence(self, data, X_label, Y_label):
    currentX_label = ''
    sequences = []
    sequence = []
    for index, row in data.iterrows():
      if currentX_label == row[X_label]:
        sequence.append(row[Y_label])
      else:
        if len(sequence) > 1:
          sequences.append(sequence)
        sequence = []
        sequence.append(row[Y_label])
      currentX_label = row[X_label]
    sequences.append(sequence)
    return sequences

  def linkAndProbabilities(self, matrices, count = 0):
    links = []
    probabilities = []
    sequences = []
    uniqueEvs = []
    for i in range(0,len(matrices)):
      thisMatrix = matrices[i]
      lastEvent = "Start";
      sequence = []
      for index, row in thisMatrix.iterrows():
        row = pd.to_numeric(row)
        #print(row) # prints the rows
        evName = row.idxmax(axis=1) # picks event with the highest probability
        link = lastEvent + "<-->" + evName
        sequence.append(evName)
        if link not in links:
          if lastEvent != evName:
            links.append(link)
            prob = row[evName] #g
            probabilities.append(prob)
        lastEvent = evName
      # The Last element is End and undesirable
      sequence.pop()
      sequences.append(sequence)
    return links, probabilities, sequences

  def linkAndProbabilities_r(self, matrices, count = 0):
    links = []
    probabilities = []
    sequences = []
    uniqueEvs = []
    for i in range(0,len(matrices)):
      thisMatrix = matrices[i]
      lastEvent = "Start";
      sequence = []
      readEvs = 0
      for index, row in thisMatrix.iterrows():
        if (count > 0 and readEvs == count):
          print(i)
          break
        row = pd.to_numeric(row)
        #print(row) # prints the rows
        evName = row.idxmax(axis=1) # picks event with the highest probability
        link = lastEvent + "<-->" + evName
        sequence.append(evName)
        if link not in links:
          if lastEvent != evName:
            links.append(link)
            prob = row[evName] #g
            probabilities.append(prob)
        lastEvent = evName
        if evName.lower() == "end":
          readEvs = readEvs + 1
      # The Last element is End and undesirable
      sequence.pop()
      sequences.append(sequence)
    return links, probabilities, sequences

  def drawGraph(self, transitions, probabilities, threshold):
    G = Digraph('process_model', filename='bpi.gv')
    G.attr(rankdir='LR', size='7,5')
    G.attr('node', shape='doublecircle', style="filled", fillcolor="grey")
    G.node('Start')
    G.node('END')
    G.attr('node', shape='box', style="bold")
    for i in range(0,len(transitions)):
      fromto = transitions[i].split("<-->")
      if (fromto[0].lower() != "end"):
        # Here Limit or Prune using the threshold
        if probabilities[i] > (threshold*100) or fromto[0].lower() == "start": # This ignores probabilty less than threshold
          G.attr('edge', style="bold", penwidth='3.0', label=str(probabilities[i]))
          G.edge(fromto[0], fromto[1])
    G.view()
    return G


In [ ]:
def getM(a, b):
  M = 0
  _index = 0;
  for index, row in a.iterrows():
    columnVal = 0
    for cell in row:
      lCell = b.iloc[_index][columnVal]
      if(lCell > 0 or cell > 0):
        M = M + 1
      columnVal = columnVal + 1
    _index = _index + 1
  return M

def getMatrix(chainTable, uniqueEV, testEvents):
  event_size = len(uniqueEV)
  matrix = np.zeros((event_size, event_size))
  counter = 0
  for strings in chainTable[0]:
    evs = strings.split("<-->")
    if testEvents is None or (evs[0] in testEvents and evs[1] in testEvents):
      probability = chainTable[1][counter]
      if probability > 1:
        probability = probability/100
      col = uniqueEV.index(evs[0])
      rw = uniqueEV.index(evs[1])
      matrix[col, rw] = probability
      dfObjio = pd.DataFrame(matrix, index=uniqueEV, columns = uniqueEV)
    counter = counter + 1
  return dfObjio

def getAdjacency(matrixOne, matrixTwo, eventCount):
  M = getM(matrixOne, matrixTwo)
  #case 1 Probability
  absoluteVal = matrixOne.subtract(matrixTwo)
  sumVal = absoluteVal.abs().sum().sum();
  #case 2 Logicals
  matrixOnelg = matrixOne.apply(np.ceil)
  matrixTwolg = matrixTwo.apply(np.ceil)
  absoluteVal = matrixOnelg.subtract(matrixTwolg)
  sumValLogical = absoluteVal.abs().sum().sum();
  #Estimate adjacency
  a = sumVal/M
  b = sumValLogical/M
  return [a, b]



In [ ]:
r = resultGraphing()
a, b = r.decomposeResult(File2)
events = list(a[0].columns.values)
events.insert(0, 'Start')
link, probabilities, sequences = r.linkAndProbabilities(a)
h1 = getMatrix([link, probabilities], events, None)

a, b = r.decomposeResult(File1)
link, probabilities, sequences = r.linkAndProbabilities(a)
h2 = getMatrix([link, probabilities], events, events)

result = getAdjacency(h1, h2, len(events))
print(result)

Draw the final graph

In [ ]:
r = resultGraphing()

In [ ]:
print(a[0])

In [ ]:
link, probabilities, sequences = r.linkAndProbabilities_r(a, 1)

In [ ]:
print(link)

In [ ]:
r.drawGraph(link, probabilities, 0)